# PINNs for Mechanical Engineers — Tutorial 2
## 2D Steady Heat Conduction in a Plate (with an exact reference)

> *In Tutorial 1 we threw a physics-informed neural network at a coupled, nonlinear problem — the lid-driven cavity — and it (mostly) worked. This time we deliberately step **down** in difficulty to a single, linear, scalar PDE. Not because it's easy content, but because it's the only heat problem where we get an **exact** answer to check against — no benchmark tables, no truncation, just a closed-form field. That makes it the perfect place to learn how PINNs handle boundary conditions honestly.*

**What you'll build:** a PINN that solves Laplace's equation on a unit square with **three Dirichlet edges and one insulated (Neumann) edge**, then validate it against the analytical separation-of-variables solution to sub-0.1% relative error.

**What you'll learn**
1. How to encode a *mixed* boundary condition set (Dirichlet **and** Neumann) as loss terms.
2. Why second derivatives need `create_graph=True` in autograd.
3. The Adam → L-BFGS two-stage recipe, and why each stage exists.
4. How to measure error against a reference field (relative $L_2$) instead of eyeballing contours.

**Honest expectation up front:** on a problem this clean, a five-line finite-difference solver will beat this PINN on both speed and accuracy. That's fine — this tutorial is about *learning the machinery on a problem where we know the right answer exactly*. The posts where PINNs actually earn their keep (coupling, inverse problems, mesh-free geometry) come later in the series.

**Runtime:** GPU-ready (NVIDIA/CUDA). On a modern GPU the full run is well under a minute. Dependencies: `torch`, `numpy`, `matplotlib` — nothing else.

## 1. The problem and its exact solution

Steady heat conduction with no internal heat source reduces to **Laplace's equation**:

$$\nabla^2 T = \frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2} = 0, \qquad (x,y)\in(0,1)\times(0,1).$$

We impose a **mixed** set of boundary conditions:

| Edge | Location | Condition | Type |
|------|----------|-----------|------|
| Left | $x=0$ | $T=0$ | Dirichlet |
| Right | $x=1$ | $T=0$ | Dirichlet |
| Bottom | $y=0$ | $\partial T/\partial y = 0$ | Neumann (insulated) |
| Top | $y=1$ | $T=\sin(\pi x)$ | Dirichlet |

**Why this exact set?** Separation of variables $T=X(x)Y(y)$ gives $X_n=\sin(n\pi x)$ (from the two zero side walls). The insulated bottom forces $Y'(0)=0$, which selects $Y_n=\cosh(n\pi y)$ (since $\cosh'(0)=\sinh(0)=0$). The top boundary $\sin(\pi x)$ is a **single Fourier mode**, so only $n=1$ survives and the infinite series collapses to one term:

$$\boxed{\,T(x,y) = \dfrac{\sin(\pi x)\,\cosh(\pi y)}{\cosh(\pi)}\,}$$

This is an **exact** reference with *zero truncation error* — any discrepancy we see is entirely the PINN's, not the reference's. That's exactly what we want for a validation-focused tutorial.

## 2. Setup

Three imports, a seed for reproducibility, and device selection. If a CUDA GPU is present it's used automatically.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math, time

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
PI = math.pi

## 3. The analytical reference

We code the exact solution first, so we always have ground truth on hand. The sanity checks confirm it satisfies the boundary conditions.

In [ ]:
def exact_solution(x, y):
    "Exact single-mode solution: T = sin(pi x) cosh(pi y) / cosh(pi)."
    return torch.sin(PI * x) * torch.cosh(PI * y) / math.cosh(PI)

# sanity checks against the boundary conditions
xs = torch.linspace(0, 1, 5).reshape(-1, 1)
print("T(0,y) =", [round(v,4) for v in exact_solution(torch.zeros_like(xs), xs).flatten().tolist()])
print("T(1,y) =", [round(v,4) for v in exact_solution(torch.ones_like(xs),  xs).flatten().tolist()])
print("T(x,1) == sin(pi x)?", torch.allclose(exact_solution(xs, torch.ones_like(xs)), torch.sin(PI*xs)))

## 4. The network

A plain multilayer perceptron: input $(x,y)$, output $T$. We use **`tanh`** activations because they are smooth and infinitely differentiable — we need to take *second* derivatives of the network output, so ReLU (whose second derivative is zero almost everywhere) would be useless here.

Four hidden layers of 64 units is comfortably more than this smooth field needs; it trains fast and leaves headroom.

In [ ]:
class PINN(nn.Module):
    def __init__(self, layers=(2, 64, 64, 64, 64, 1)):
        super().__init__()
        seq = []
        for i in range(len(layers) - 1):
            lin = nn.Linear(layers[i], layers[i+1])
            nn.init.xavier_normal_(lin.weight)
            nn.init.zeros_(lin.bias)
            seq.append(lin)
            if i < len(layers) - 2:
                seq.append(nn.Tanh())
        self.net = nn.Sequential(*seq)
    def forward(self, x, y):
        return self.net(torch.cat([x, y], dim=1))

model = PINN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Trainable parameters: {n_params}")

## 5. Collocation and boundary points

PINNs are **mesh-free**: instead of a grid, we sample points where we'll enforce the equations.

- **Interior (collocation) points** — where we enforce the PDE residual $\nabla^2 T = 0$. These need `requires_grad=True` because we differentiate the network at them.
- **Boundary points** — where we enforce the four edge conditions. The insulated bottom edge also needs gradients (it's a derivative condition).

We seed a separate generator so the point cloud is reproducible. (The top and bottom edges share the same $x$-coordinates — harmless, and keeps the sampling compact.)

In [ ]:
gen = torch.Generator().manual_seed(42)   # reproducible point cloud
N_f = 2000    # interior collocation points
N_b = 200     # points per boundary edge

def rand_col(n):
    return torch.rand(n, 1, generator=gen)

# interior points (need grads for the PDE residual)
x_f = rand_col(N_f).to(device).requires_grad_(True)
y_f = rand_col(N_f).to(device).requires_grad_(True)

# left (x=0) and right (x=1): share y-coordinates
y_lr    = rand_col(N_b).to(device)
x_left  = torch.zeros(N_b, 1, device=device)
x_right = torch.ones (N_b, 1, device=device)

# top (y=1) and bottom (y=0): share x-coordinates
x_tb  = rand_col(N_b).to(device)
y_top = torch.ones (N_b, 1, device=device)
T_top = torch.sin(PI * x_tb)                       # top Dirichlet target
y_bot = torch.zeros(N_b, 1, device=device).requires_grad_(True)  # Neumann -> needs grad

print(f"interior points: {N_f}, boundary points per edge: {N_b}")

## 6. The PDE residual via autograd

The heart of a PINN. We use `torch.autograd.grad` to differentiate the network output with respect to its inputs. The key flag is **`create_graph=True`**: it keeps the computational graph *alive* after the first derivative so we can differentiate again to get the second derivative. Forget it, and your second derivatives silently come out as `None`/zero.

In [ ]:
def grad(outputs, inputs):
    "First derivative d(outputs)/d(inputs), graph retained for higher-order diff."
    return torch.autograd.grad(
        outputs, inputs,
        grad_outputs=torch.ones_like(outputs),
        create_graph=True)[0]

# quick demo on the untrained network
T   = model(x_f, y_f)
T_x = grad(T, x_f); T_xx = grad(T_x, x_f)
T_y = grad(T, y_f); T_yy = grad(T_y, y_f)
print("residual shape:", tuple((T_xx + T_yy).shape),
      "| mean|r| (untrained): %.3e" % (T_xx + T_yy).abs().mean().item())

## 7. The composite loss

The total loss is the PDE residual plus one term per boundary condition:

$$\mathcal{L} = \underbrace{\big\lVert T_{xx}+T_{yy} \big\rVert^2}_{\text{PDE}}
+ \lambda\Big(\underbrace{\lVert T\rVert^2_{x=0} + \lVert T\rVert^2_{x=1}}_{\text{side walls}}
+ \underbrace{\lVert T-\sin\pi x\rVert^2_{y=1}}_{\text{top}}
+ \underbrace{\lVert T_y\rVert^2_{y=0}}_{\text{insulated}}\Big)$$

The Neumann term is just the mean-squared **normal derivative** on the bottom edge — enforcing $\partial T/\partial y = 0$ is as simple as penalising `grad(T, y)` there. We weight the boundary terms by $\lambda=10$ so the network doesn't satisfy the interior equation while drifting on the edges (a common failure mode — see the table at the end).

In [ ]:
BC_WEIGHT = 10.0

def compute_losses():
    # PDE residual (Laplace) on interior points
    T = model(x_f, y_f)
    T_xx = grad(grad(T, x_f), x_f)
    T_yy = grad(grad(T, y_f), y_f)
    loss_pde = (T_xx + T_yy).pow(2).mean()

    # Dirichlet edges
    loss_left  = model(x_left,  y_lr).pow(2).mean()            # T=0
    loss_right = model(x_right, y_lr).pow(2).mean()            # T=0
    loss_top   = (model(x_tb, y_top) - T_top).pow(2).mean()    # T=sin(pi x)

    # Neumann (insulated) bottom edge: dT/dy = 0
    loss_bot = grad(model(x_tb, y_bot), y_bot).pow(2).mean()

    loss_bc = loss_left + loss_right + loss_top + loss_bot
    return loss_pde, loss_bc

def total_loss():
    lp, lb = compute_losses()
    return lp + BC_WEIGHT * lb

## 8. Error metric

We judge the PINN by the **relative $L_2$ error** against the exact field on a dense $200\times200$ grid — a single honest number, not a contour we squint at.

In [ ]:
n_grid = 200
gx = torch.linspace(0, 1, n_grid)
GX, GY = torch.meshgrid(gx, gx, indexing='ij')
xg = GX.reshape(-1, 1).to(device)
yg = GY.reshape(-1, 1).to(device)
T_exact = exact_solution(xg, yg)

def rel_L2_error():
    with torch.no_grad():
        T_pred = model(xg, yg)
    return (torch.linalg.norm(T_pred - T_exact) /
            torch.linalg.norm(T_exact)).item()

print(f"relative L2 error (untrained): {rel_L2_error():.3e}")

## 9. Stage 1 — Adam

Adam gets us into the right basin quickly. We log the PDE loss, boundary loss, and relative $L_2$ error every 500 iterations.

**Reference run (this exact code, seeds fixed):**
```
[Adam]     0  loss=9.101e+00  pde=7.387e-03  bc=9.094e-01  relL2=1.434e+00
[Adam]   500  loss=6.024e-01  pde=1.521e-01  bc=4.503e-02  relL2=2.126e-01
[Adam]  1000  loss=4.967e-02  pde=2.041e-02  bc=2.926e-03  relL2=4.512e-02
[Adam]  1500  loss=1.138e-02  pde=7.032e-03  bc=4.347e-04  relL2=1.712e-02
[Adam]  2000  loss=4.190e-03  pde=3.139e-03  bc=1.051e-04  relL2=6.605e-03
[Adam]  2500  loss=2.290e-03  pde=1.632e-03  bc=6.575e-05  relL2=5.072e-03
[Adam]  3000  loss=1.838e-03  pde=1.086e-03  bc=7.523e-05  relL2=8.223e-03
```
Notice the relative error *wobbles* between iters 2500 and 3000 — Adam is now bouncing around the minimum rather than descending into it. That's precisely the signal that it's time to switch optimizers.

In [ ]:
history = []   # (iteration, total_loss) for the loss curve

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
ADAM_ITERS = 3000
t0 = time.time()
for it in range(ADAM_ITERS + 1):
    optimizer.zero_grad()
    loss = total_loss()
    loss.backward()
    optimizer.step()
    if it % 500 == 0:
        lp, lb = compute_losses()
        history.append((it, loss.item()))
        print(f"[Adam] {it:5d}  loss={loss.item():.3e}  "
              f"pde={lp.item():.3e}  bc={lb.item():.3e}  relL2={rel_L2_error():.3e}")
print(f"Adam finished in {time.time()-t0:.1f}s")

## 10. Stage 2 — L-BFGS refinement

L-BFGS is a quasi-Newton method: it uses curvature information and a line search, which lets it drive smooth-problem residuals *far* lower than Adam can. This is where the accuracy actually comes from.

**One pitfall worth flagging** (it has bitten this series before): the L-BFGS `closure` must rebuild the graph and call `backward()` on **every** invocation. Do **not** wrap the forward pass in `torch.no_grad()` — L-BFGS calls the closure multiple times per step during its line search, and a `no_grad` context leaves it with no gradient to work with, producing silent stalls or `NaN`s.

**Reference run:**
```
[L-BFGS]  100  loss=4.493e-04  pde=3.903e-04  bc=5.898e-06  relL2=1.789e-03
[L-BFGS]  200  loss=2.368e-04  pde=2.050e-04  bc=3.178e-06  relL2=1.222e-03
[L-BFGS]  300  loss=1.067e-04  pde=7.772e-05  bc=2.895e-06  relL2=1.201e-03
[L-BFGS]  400  loss=7.542e-05  pde=4.916e-05  bc=2.626e-06  relL2=1.197e-03
[L-BFGS]  500  loss=5.741e-05  pde=3.801e-05  bc=1.941e-06  relL2=1.027e-03
```
An order of magnitude off the loss and roughly $8\times$ off the error, in a fraction of the wall-clock Adam took.

In [ ]:
optimizer = torch.optim.LBFGS(
    model.parameters(), lr=1.0, max_iter=500, max_eval=600,
    tolerance_grad=1e-9, tolerance_change=1e-12,
    history_size=50, line_search_fn='strong_wolfe')

base, counter = ADAM_ITERS, [0]
def closure():
    optimizer.zero_grad()
    loss = total_loss()        # NB: no torch.no_grad() here!
    loss.backward()
    counter[0] += 1
    if counter[0] % 100 == 0:
        lp, lb = compute_losses()
        history.append((base + counter[0], loss.item()))
        print(f"[L-BFGS] {counter[0]:4d}  loss={loss.item():.3e}  "
              f"pde={lp.item():.3e}  bc={lb.item():.3e}  relL2={rel_L2_error():.3e}")
    return loss

t0 = time.time()
optimizer.step(closure)
print(f"L-BFGS finished in {time.time()-t0:.1f}s ({counter[0]} evaluations)")

## 11. Final error

**Reference run:** `relative L2 = 9.67e-04`, `max abs error = 1.98e-03`. Sub-0.1% against an exact field — a clean validation.

In [ ]:
with torch.no_grad():
    T_pred = model(xg, yg)
print(f"Final relative L2 error : {rel_L2_error():.3e}")
print(f"Final max abs error     : {(T_pred - T_exact).abs().max().item():.3e}")

## 12. Prediction vs. exact

Three panels: the PINN field, the exact field, and their absolute difference. Watch where the error lives.

In [ ]:
with torch.no_grad():
    Tp = model(xg, yg).reshape(n_grid, n_grid).cpu().numpy()
Te = T_exact.reshape(n_grid, n_grid).cpu().numpy()
Er = np.abs(Tp - Te)
Xp, Yp = GX.numpy(), GY.numpy()

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a, (D, ttl, cmap) in zip(ax, [(Tp,'PINN prediction','viridis'),
                                  (Te,'Exact (analytical)','viridis'),
                                  (Er,'Absolute error','inferno')]):
    cf = a.contourf(Xp, Yp, D, levels=50, cmap=cmap)
    fig.colorbar(cf, ax=a, fraction=0.046)
    a.set_title(ttl); a.set_xlabel('x'); a.set_ylabel('y'); a.set_aspect('equal')
plt.tight_layout(); plt.show()

The error piles up in the **bottom two corners** — where the $T=0$ side walls meet the insulated base. Corners where different boundary-condition *types* collide are the hardest thing for the network to represent, and they're where you'll almost always see a PINN's residual concentrate. Physically, the field is exactly what you'd expect: the hot $\sin(\pi x)$ profile on top diffuses downward and flattens as it approaches the insulated floor.

In [ ]:
its = [h[0] for h in history]; ls = [h[1] for h in history]
plt.figure(figsize=(7, 4))
plt.semilogy(its, ls, 'o-', ms=3)
plt.axvline(ADAM_ITERS, ls='--', c='gray', label='Adam → L-BFGS')
plt.xlabel('iteration'); plt.ylabel('total loss')
plt.title('Training loss'); plt.legend(); plt.tight_layout(); plt.show()

## 13. What just happened?

| Stage | iterations | total loss | relative $L_2$ |
|-------|-----------:|-----------:|---------------:|
| start (untrained) | 0 | 9.1e+00 | 1.43 |
| after Adam | 3000 | 1.84e-03 | 8.2e-03 |
| after L-BFGS | +539 | 5.7e-05 | **9.7e-04** |

The whole story in one line: **Adam finds the basin, L-BFGS polishes the bottom of it.** We enforced a genuinely mixed boundary set — three Dirichlet edges as value-matching terms, one insulated edge as a derivative-matching term — and recovered an exact analytical field to sub-0.1%. The Neumann condition cost us exactly one extra line of code.

And the honest footnote: a textbook finite-difference stencil solves this same problem to machine precision in milliseconds. The PINN's value isn't here — it's in the problems where a mesh is painful or the physics is only partially known. This tutorial was about trusting the machinery on a problem where we can *check every digit*.

## 14. Failure modes to watch for

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Loss stalls around $10^{-1}$, field looks flat | boundary terms too weak, or net too small | raise `BC_WEIGHT`; widen/deepen the net |
| Relative error *wanders* late in Adam | Adam bouncing around the minimum | that's your cue to switch to L-BFGS |
| L-BFGS returns `NaN` or doesn't move | forward pass wrapped in `torch.no_grad()`, or no line search | rebuild the graph in the closure; use `strong_wolfe` |
| Second derivatives come out zero | forgot `create_graph=True` | set it in the `grad` helper |
| Error concentrates at corners | Dirichlet/Neumann clash at the corner | add boundary points; accept it — it's inherent |

## 15. Exercises

1. **Multi-mode top BC.** Change the top boundary to $T(x,1)=\sin(2\pi x)+\tfrac12\sin(3\pi x)$. The exact solution is now a *two-term* series $\sum_n c_n \sin(n\pi x)\cosh(n\pi y)$ with $c_n = [\text{coeff}]/\cosh(n\pi)$. Code the reference and re-measure the error.
2. **Robin (convective) bottom.** Replace the insulated base with a convective condition $-k\,\partial T/\partial y = h\,(T - T_\infty)$. This is the most common real-world boundary in heat transfer — add it as a loss term and pick your own $k,h,T_\infty$.
3. **Hard constraints.** Enforce the two side walls *exactly* with an ansatz $T = x(1-x)\,N(x,y) + \text{(lift for the top)}$ so those BC losses vanish by construction. Does it converge faster or more accurately than the soft-constraint version here?
4. **Poisson with a source.** Add a heat source: $\nabla^2 T = -q/k$. Use the method of manufactured solutions — pick a target $T$, differentiate it to get the required $q$, then check the PINN recovers it.
5. **Adam-only challenge.** Turn off L-BFGS. How many Adam iterations does it take to reach $10^{-3}$ relative error — if it gets there at all?

## 16. Vocabulary

- **Collocation point** — an interior sample where the PDE residual is enforced (the mesh-free analogue of a grid node).
- **PDE residual** — how badly the network violates the governing equation, here $T_{xx}+T_{yy}$.
- **Dirichlet BC** — prescribes the *value* of the field on a boundary.
- **Neumann BC** — prescribes the *normal derivative*; $\partial T/\partial n = 0$ is a perfectly insulated wall.
- **Robin / convective BC** — a mix of value and derivative, e.g. Newton cooling.
- **Soft vs. hard constraint** — enforcing a BC via a loss penalty (soft) vs. building it into the network architecture so it holds exactly (hard).
- **`create_graph=True`** — keeps the autograd graph alive so you can take a derivative *of* a derivative.
- **Relative $L_2$ error** — $\lVert T_\text{pred}-T_\text{exact}\rVert_2 / \lVert T_\text{exact}\rVert_2$, a scale-free accuracy measure.
- **L-BFGS** — a quasi-Newton optimizer; excels at polishing smooth low-residual problems.

## 17. What's next

**Tutorial 3 — Linear elasticity: a plate with a central hole.** We move from a scalar field to a **vector** output $(u,v)$ displacements, recover **stress** by autograd-differentiating the displacement field, and validate against the classic **Kirsch** analytical solution — including the famous stress-concentration factor of **3** at the hole edge. First coupled equilibrium system of the series, and the contours make a genuinely satisfying picture.